In [1]:
!pip install pymongo

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 7.9 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 6.8 MB/s  0:00:00

   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   -------------------- ------------------- 1/2 [pymongo]
   ---------------------------------------- 2/2 [pymongo]



In [113]:
from decimal import Decimal
from bson.decimal128 import Decimal128
import sqlalchemy as sa
from config import POSTGRES_USER, POSTGRES_PASS, POSTGRES_DBNAME, POSTGRES_HOST, POSTGRES_PORT
from config import MONGO_USER, MONGO_PASS, CLUSTER
from urllib.parse import quote_plus
import pymongo
user = quote_plus(MONGO_USER)
password = quote_plus(MONGO_PASS)

client = pymongo.MongoClient(f"mongodb+srv://{user}:{password}@{CLUSTER}/?appName=PostgresToMongo")

db = client['chinook_first_200_id']

DATABASE_URL = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASS}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DBNAME}"

engine = sa.create_engine(DATABASE_URL)

tables_to_mongo = ['track', 'genre', 'album', 'media_type', 'artist']

with engine.connect() as conn:
    conn.execute(sa.text(f"CREATE TEMP TABLE tracks_below_id_200 AS "
                 f"SELECT * FROM track WHERE track_id <= 200"))

    for table in tables_to_mongo:
        if table != "artist":
            query = f"SELECT DISTINCT t.* FROM {table} t JOIN tracks_below_id_200 ON t.{table}_id = tracks_below_id_200.{table}_id"
        else:
            query = f"""SELECT DISTINCT t.* FROM {table} t
                        JOIN album alb ON t.{table}_id = alb.{table}_id
                        JOIN tracks_below_id_200 tb ON tb.album_id = alb.album_id
                        """
        data = [dict(c) for c in conn.execute(sa.text(query)).mappings()]
        if data:
            db[table].drop()
            for row in data:
                for key, value in row.items():
                    if isinstance(value, Decimal):
                        row[key] = Decimal128(str(value))
            db[table].insert_many(data)

In [14]:

client.list_database_names()

['chinook_first_200_id', 'sample_mflix', 'admin', 'local']

In [54]:
rock_id = db.genre.distinct('genre_id', {'name': 'Rock'})
rock_track = list(db.track.find({'genre_id': {'$in': list(rock_id)}}))
rock_track_name = "\n".join(db.track.distinct('name', {'genre_id': {'$in': rock_id}}))
print(f""" Знайдено {db.track.count_documents({'genre_id': {'$in': rock_id}})} треків з жанром Rock,
документи з повною інформацією по цим трекам можна отримати із змінної 'rock_track'.
Ось назви цих треків:
{rock_track_name}, """)



 Знайдено 76 треків з жанром Rock,
документи з повною інформацією по цим трекам можна отримати із змінної 'rock_track'.
Ось назви цих треків:
All I Really Want
Amazing
Angel
Bad Boy Boogie
Balls to the Wall
Bleed The Freak
Blind Man
Breaking The Rules
Bring'em Back Alive
C.O.D.
Cochise
Confusion
Crazy
Cryin'
Deuces Are Wild
Dog Eat Dog
Dude (Looks Like A Lady)
Eat The Rich
Evil Walks
Exploder
Fast As a Shark
For Those About To Rock (We Salute You)
Forgiven
Gasoline
Getaway Car
Go Down
Hand In My Pocket
Head Over Feet
Hell Ain't A Bad Place To Be
Hypnotize
I Can't Remember
I Know Somethin (Bout You)
I am the Highway
Inject The Venom
Ironic
It Ain't Like That
Janie's Got A Gun
Let There Be Rock
Let's Get It Up
Light My Way
Like a Stone
Livin' On The Edge
Love In An Elevator
Love, Hate, Love
Man In The Box
Mary Jane
Night Of The Long Knives
Not The Doctor
Overdose
Perfect
Princess of the Dawn
Problem Child
Put The Finger On You
Put You Down
Rag Doll
Real Thing
Restless and Wild
Right Thro

In [85]:
AC_DC_id = db.artist.distinct('artist_id', {'name': 'AC/DC'})
AC_DC_album = db.album.distinct('title', {'artist_id': AC_DC_id[0]})
print(AC_DC_album)

['For Those About To Rock We Salute You', 'Let There Be Rock']


In [103]:

print(list(db.track.aggregate([
    {
        '$lookup': {
            'from': "genre",
            'localField': "genre_id",
            'foreignField': "genre_id",
            'as': "track_genre"
        }
    },
    {
        '$unwind': '$track_genre'
    },
    {
        '$group': {
            '_id': '$name',
            'genre': {'$first': '$track_genre.name'}
        }
    },
    {
        '$project': {
            '_id': 0,
            'track_name': '$_id',
            'genre_name': '$genre'
        }
    },

])))


[{'track_name': 'Gates Of Urizen', 'genre_name': 'Metal'}, {'track_name': 'Smoked Pork', 'genre_name': 'Alternative & Punk'}, {'track_name': 'Oprah', 'genre_name': 'Alternative & Punk'}, {'track_name': 'Black Sabbath', 'genre_name': 'Metal'}, {'track_name': 'Rag Doll', 'genre_name': 'Rock'}, {'track_name': '20 Flight Rock', 'genre_name': 'Rock And Roll'}, {'track_name': 'Bowels Of The Devil', 'genre_name': 'Alternative & Punk'}, {'track_name': 'Welcome Home (Sanitarium)', 'genre_name': 'Metal'}, {'track_name': 'Samba De Uma Nota Só (One Note Samba)', 'genre_name': 'Jazz'}, {'track_name': 'Love In An Elevator', 'genre_name': 'Rock'}, {'track_name': 'Eat The Rich', 'genre_name': 'Rock'}, {'track_name': 'Bleed The Freak', 'genre_name': 'Rock'}, {'track_name': "Rock 'N' Roll Music", 'genre_name': 'Rock And Roll'}, {'track_name': 'Slow Down', 'genre_name': 'Rock And Roll'}, {'track_name': 'Problem Child', 'genre_name': 'Rock'}, {'track_name': "Bring'em Back Alive", 'genre_name': 'Rock'}, {'

In [120]:
print(list(db.track.aggregate([
    {
        '$lookup': {
            'from': "genre",
            'localField': "genre_id",
            'foreignField': "genre_id",
            'as': "genre_info"
        }
    },
    {
        '$unwind': '$genre_info'
    },
    {
        '$group': {
            '_id': '$genre_info.name',
            'count': { '$sum': 1 }
        }
    },
    {
        '$project': {
            '_id': 0,
            'genre': '$_id',
            'total_tracks': '$count'
        }
    }
])))

[{'genre': 'Metal', 'total_tracks': 54}, {'genre': 'Alternative & Punk', 'total_tracks': 29}, {'genre': 'Jazz', 'total_tracks': 22}, {'genre': 'Rock And Roll', 'total_tracks': 12}, {'genre': 'Rock', 'total_tracks': 76}, {'genre': 'Blues', 'total_tracks': 7}]


In [123]:
genre_info = db.genre.find_one({'name': 'Alternative & Punk'})

tracks = db.track.find({'genre_id': genre_info['genre_id']}) \
                     .sort('milliseconds', -1) \
                     .limit(5)

print("5 найдовших треків (Alternative & Punk):")
for t in tracks:
    ms = t['milliseconds']
    mins = ms // 60000
    secs = (ms % 60000) // 1000
    print(f"- {t['name']}: {mins:02d}:{secs:02d}")

5 найдовших треків (Alternative & Punk):
- The Winner Loses: 06:32
- Momma's Gotta Die Tonight: 06:11
- There Goes The Neighborhood: 05:50
- Body Count: 05:17
- The Curse: 05:09
